# PROTAC run analysisLinker-focused view of a `mode: protac` run. The question here is not only "did it score well" but "are these twenty linkers actually different, or twenty PEGs of different length".

In [ ]:
from pathlib import Pathimport jsonimport pandas as pdimport matplotlib.pyplot as pltfrom molgen.chem.protac import classify_linker, linker_descriptors, protac_property_windowRUN_DIR = Path("../runs/protac_demo")run = json.loads((RUN_DIR / "run.json").read_text())df = pd.read_csv(RUN_DIR / "results.csv")print(f"candidates {len(df)}  |  valid {int(df['valid'].sum())}")df.head()

## 1. Linker chemotype spreadAll one class is the failure to look for.

In [ ]:
valid = df[df["valid"] == True].copy()valid["linker_class"] = valid["canonical_smiles"].map(    lambda s: classify_linker(s) if isinstance(s, str) else "unknown")counts = valid["linker_class"].value_counts()fig, ax = plt.subplots(figsize=(7, 3.5))ax.bar(counts.index, counts.values, color="#8e6bb5")ax.set_ylabel("count")ax.set_title("Linker classes")plt.tight_layout()plt.show()print(counts.to_string())if len(counts) == 1:    print("\nSingle class across the whole set — raise temperature or strengthen the variety instruction.")

## 2. Degrader property windowRule-of-five does not apply. These bounds are advisory.

In [ ]:
window = pd.DataFrame([    {**{"name": row["name"]}, **protac_property_window(row["canonical_smiles"])}    for _, row in valid.iterrows()    if isinstance(row["canonical_smiles"], str)])if not window.empty:    inside = int(window["in_window"].sum())    print(f"{inside}/{len(window)} inside the degrader window (MW 700-1100, cLogP 2-7, TPSA < 250)")    fig, axes = plt.subplots(1, 3, figsize=(15, 4))    for ax, (col, lo, hi, label) in zip(axes, [        ("mw", 700, 1100, "MW (Da)"), ("logp", 2, 7, "cLogP"), ("tpsa", 0, 250, "TPSA"),    ]):        ax.hist(window[col].dropna(), bins=18, color="#8e6bb5", edgecolor="white")        ax.axvline(lo, color="#c0392b", linestyle="--", linewidth=1)        ax.axvline(hi, color="#c0392b", linestyle="--", linewidth=1)        ax.set_xlabel(label)    plt.tight_layout()    plt.show()

## 3. Linker length vs scoreProductive ternary complexes are usually length-sensitive. A visible optimum is the most actionable signal a PROTAC run produces.

In [ ]:
scored = valid.dropna(subset=["best_affinity"]) if "best_affinity" in valid else valid.iloc[0:0]if not scored.empty:    fig, ax = plt.subplots(figsize=(7, 4.5))    ax.scatter(scored["rotatable_bonds"], scored["best_affinity"], alpha=.7, color="#8e6bb5")    ax.set_xlabel("rotatable bonds (linker flexibility proxy)")    ax.set_ylabel("Vina score (kcal/mol)")    ax.set_title("Flexibility vs score")    plt.tight_layout()    plt.show()else:    print("no docking results in this run")

## 4. Top degraders

In [ ]:
from rdkit import Chemfrom rdkit.Chem import Drawbest = scored.nsmallest(8, "best_affinity") if not scored.empty else valid.head(8)mols, legends = [], []for _, row in best.iterrows():    mol = Chem.MolFromSmiles(row["canonical_smiles"])    if mol is None:        continue    mols.append(mol)    score = row.get("best_affinity")    legends.append(f"{row['name']}\n{score:.2f}" if pd.notna(score) else str(row["name"]))Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(420, 260), legends=legends)